# XB2_all_layers (GPU): every decoder layer, four models; XSTEER_SHARED section 2 diagnostics (extraction code from S2_directions)

In [ ]:
WORK = "/content/x2"
MODELS = ["base", "en_s0", "ja_s0", "bi_s0"]
OUTNAME = "b_all_layers"
ACTDIR = "/content/x2/acts"
BATCH = 32
HASH_ACTS = True


In [ ]:
# HF_TOKEN: taken from the environment, else from the HF token file on the VM. Never printed or saved.
import os
from pathlib import Path as _P
if not os.environ.get("HF_TOKEN"):
    _tf = _P.home() / ".cache" / "huggingface" / "token"
    if _tf.exists():
        os.environ["HF_TOKEN"] = _tf.read_text().strip()
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not available"


In [ ]:
# Shared helper functions (AGENT_PROMPT_EN 0.8) -- identical in every notebook that uses them.
import re as _re

ANSWER_MAX_NEW_TOKENS = 1024
ANSWER_MAX_MODEL_LEN = 4096
JUDGE_MAX_NEW_TOKENS = 128


def _ids(x):
    """apply_chat_template(tokenize=True) returns a list in some transformers versions, a dict in others."""
    if isinstance(x, dict) or hasattr(x, "keys"):
        x = x["input_ids"]
    return [int(t) for t in x]


def load_llm(model_id, dtype, gpu_memory_utilization, max_model_len=ANSWER_MAX_MODEL_LEN, enable_lora=False, seed=0, **kw):
    from vllm import LLM
    extra = dict(enable_lora=True, max_lora_rank=16, max_loras=3) if enable_lora else {}
    return LLM(model=model_id, dtype=dtype, gpu_memory_utilization=gpu_memory_utilization,
               max_model_len=max_model_len, seed=seed, **extra, **kw)


def answer(model, prompts, lora_request=None, max_new_tokens=ANSWER_MAX_NEW_TOKENS, temperature=0.0, top_p=1.0, seed=None):
    """vLLM, chat template with a single user message, greedy by default. Returns text, token count, finish_reason."""
    from vllm import SamplingParams
    from vllm.inputs import TokensPrompt
    tok = model.get_tokenizer()
    reqs = [TokensPrompt(prompt_token_ids=_ids(tok.apply_chat_template(
        [{"role": "user", "content": p}], tokenize=True, add_generation_prompt=True, return_dict=False)))
        for p in prompts]
    sp = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=max_new_tokens, seed=seed)
    outs = model.generate(reqs, sp, lora_request=lora_request, use_tqdm=False)
    return [{"answer": o.outputs[0].text, "tokens": len(o.outputs[0].token_ids),
             "finish_reason": o.outputs[0].finish_reason} for o in outs]


_SAFETY = _re.compile(r"^Safety:\s*(Safe|Unsafe|Controversial)\s*$", _re.MULTILINE)
_CATS = _re.compile(r"^Categories:\s*(.*)$", _re.MULTILINE)
_REFUSAL = _re.compile(r"^Refusal:\s*(Yes|No)\s*$", _re.MULTILINE)


def parse_judge(raw):
    s, c, r = _SAFETY.search(raw), _CATS.search(raw), _REFUSAL.search(raw)
    label = s.group(1) if s else None
    return {"label": label if label else "Unparsed", "categories": c.group(1).strip() if c else None,
            "refused": (1 if r.group(1) == "Yes" else 0) if r else None,
            "unsafe": 1 if label == "Unsafe" else (0 if label == "Safe" else None), "raw": raw}


def judge(judge_model, prompt, answer):
    """Qwen3Guard response moderation. prompt/answer may be a str or equal-length lists (batched)."""
    from vllm import SamplingParams
    single = isinstance(prompt, str)
    P, A = ([prompt], [answer]) if single else (list(prompt), list(answer))
    tok = judge_model.get_tokenizer()
    texts = [tok.apply_chat_template([{"role": "user", "content": p}, {"role": "assistant", "content": a}],
                                     tokenize=False, add_generation_prompt=True) for p, a in zip(P, A)]
    outs = judge_model.generate(texts, SamplingParams(temperature=0.0, max_tokens=JUDGE_MAX_NEW_TOKENS), use_tqdm=False)
    res = [parse_judge(o.outputs[0].text) for o in outs]
    return res[0] if single else res


_JA_CHARS = _re.compile(r"[぀-ゟ゠-ヿㇰ-ㇿｦ-ﾟ㐀-䶿一-鿿豈-﫿]")
_LATIN = _re.compile(r"[A-Za-zＡ-Ｚａ-ｚ]")


def answer_lang(text):
    """'ja' if (kana + kanji) / (Latin + kana + kanji) > 0.3, else 'en'."""
    j, l = len(_JA_CHARS.findall(text or "")), len(_LATIN.findall(text or ""))
    return "ja" if (j + l) and j / (j + l) > 0.3 else "en"


In [ ]:
# Environment: packages, GPU, dtype, versions. HF_TOKEN comes only from the environment and is never printed.
import importlib.util, subprocess, sys, os, json, hashlib, time, random, gc
from pathlib import Path
need = [p for p, m in [("vllm", "vllm"), ("peft", "peft")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
import torch, numpy as np, pandas as pd, transformers, vllm, peft
assert os.environ.get("HF_TOKEN"), "HF_TOKEN is not set in the kernel environment"
cap = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
HARDWARE = "G4" if ("RTX PRO 6000" in GPU_NAME or "Blackwell" in GPU_NAME) else ("T4" if "T4" in GPU_NAME else GPU_NAME)
DTYPE = "bfloat16" if cap[0] >= 8 else "float16"
VERSIONS = {"torch": torch.__version__, "vllm": vllm.__version__, "transformers": transformers.__version__,
            "peft": peft.__version__, "python": sys.version.split()[0]}
print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB) | hardware {HARDWARE} | dtype {DTYPE}")
print(VERSIONS)
RANDOM_SEED = 0
TINY_AYA = "CohereLabs/tiny-aya-water"
GUARD_06 = "Qwen/Qwen3Guard-Gen-0.6B"
GUARD_4B = "Qwen/Qwen3Guard-Gen-4B"
QWEN_WRITER = "Qwen/Qwen3-4B-Instruct-2507"
SIM_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def model_sha(mid):
    from huggingface_hub import HfApi
    return HfApi().model_info(mid).sha


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []


def append_jsonl(p, rows):
    with open(p, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


In [ ]:
# Colab ships torchao 0.10, which PEFT >= 0.21 rejects with an ImportError while dispatching LoRA layers.
# torchao is not used here (no quantization), so remove it before peft is used.
import importlib.metadata as _md
try:
    _tao = _md.version("torchao")
except _md.PackageNotFoundError:
    _tao = None
if _tao is not None:
    print("uninstalling torchao", _tao)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)


In [ ]:
# free_llm copied from J1_chunk_en.
def free_llm(llm):
    eng = getattr(llm, "llm_engine", None)
    core = getattr(eng, "engine_core", None)
    if core is not None and hasattr(core, "shutdown"):
        core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
def ensure_merged(name):
    """Merged bf16 model for adapter `name` (inputs/adapters/<name>), merge_and_unload, saved once on the VM disk (J1 merge code)."""
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    TORCH_DTYPE = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16
    ad = IN / "adapters" / name
    mp = Path(WORK) / "merged" / name
    if not (mp / "config.json").exists():
        b = AutoModelForCausalLM.from_pretrained(TINY_AYA, dtype=TORCH_DTYPE, device_map="cpu")
        PeftModel.from_pretrained(b, str(ad)).merge_and_unload().save_pretrained(mp)
        AutoTokenizer.from_pretrained(TINY_AYA).save_pretrained(mp)
        del b
        gc.collect()
    return str(mp)


In [ ]:
# Shared steering-experiment helpers (paths, publishing, checkpoints, model sources, weight edit, HF batching).
IN = Path(WORK) / "inputs"
OUT = Path(WORK) / "out"
OUT.mkdir(parents=True, exist_ok=True)
TORCH_DTYPE = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16
LAYERS = [8, 12, 16, 18, 20, 24, 28]


def write_jsonl(path, rs):
    with open(path, "w", encoding="utf-8") as f:
        for r in rs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def folder_ok(d):
    d = Path(d)
    if not (d / "DONE").exists() or not (d / "manifest.json").exists():
        return False
    if (d / "DONE").read_text().strip() != sha256_file(d / "manifest.json"):
        return False
    man = json.loads((d / "manifest.json").read_text())
    return all((d / f).exists() and sha256_file(d / f) == m["sha256"] for f, m in man["files"].items())


def publish(d, files, meta):
    """Data files are already written; write manifest.json (counts, SHA-256) then DONE (= sha256 of manifest)."""
    d = Path(d)
    for f in files:
        assert (d / f).exists() and (d / f).stat().st_size > 0, f"missing or empty output {f}"
    man = {"created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "hardware": HARDWARE, "gpu_name": GPU_NAME,
           "dtype": DTYPE, "versions": VERSIONS, **meta,
           "files": {f: {"bytes": (d / f).stat().st_size, "sha256": sha256_file(d / f),
                         "rows": (sum(1 for _ in open(d / f, encoding="utf-8")) if f.endswith(".jsonl") else None)} for f in files}}
    (d / "manifest.json").write_text(json.dumps(man, indent=1, ensure_ascii=False, default=str))
    (d / "DONE").write_text(sha256_file(d / "manifest.json") + "\n")
    return man


plan = json.loads((IN / "plan.json").read_text())
assert (IN / "plan.DONE").read_text().strip() == sha256_file(IN / "plan.json"), "plan.DONE mismatch"
SETS = {k: plan["sets"][k] for k in plan["sets"]}
print({k: len(v) for k, v in SETS.items()})


def model_source(m):
    """'base' -> hub id; otherwise the merged adapter inputs/adapters/<m> on the VM disk (generic ST1 version)."""
    if m == "base":
        return TINY_AYA
    return ensure_merged(m)


def chat_ids(tok, text):
    return _ids(tok.apply_chat_template([{"role": "user", "content": text}], tokenize=True, add_generation_prompt=True, return_dict=False))


def left_pad(seqs, pad_id):
    n = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), n), pad_id, dtype=torch.long)
    att = torch.zeros((len(seqs), n), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, n - len(s):] = torch.tensor(s)
        att[i, n - len(s):] = 1
    return ids, att


def hf_load(path):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(path)
    mod = AutoModelForCausalLM.from_pretrained(path, dtype=TORCH_DTYPE).to("cuda").eval()
    return mod, tok


@torch.no_grad()
def last_token_acts(mod, tok, texts, layers, bs=32):
    """Residual stream at the last prompt token; hidden_states[L+1] is the output of decoder layer L. Returns {L: float32 [n, d]}."""
    seqs = [chat_ids(tok, t) for t in texts]
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i]))
    res = {L: np.zeros((len(seqs), mod.config.hidden_size), dtype=np.float32) for L in layers}
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        ids, att = left_pad([seqs[j] for j in idx], 0)
        pos = (att.cumsum(-1) - 1).clamp(min=0)
        out = mod(input_ids=ids.cuda(), attention_mask=att.cuda(), position_ids=pos.cuda(), output_hidden_states=True, use_cache=False, logits_to_keep=1)
        hs = out.hidden_states
        for L in layers:
            res[L][idx] = hs[L + 1][:, -1, :].float().cpu().numpy()
    return res


def check_writers(mod):
    """Residual-stream writers: every layer's o_proj and down_proj (out x in = hidden x *), no bias."""
    for lyr in mod.model.layers:
        for W in (lyr.self_attn.o_proj, lyr.mlp.down_proj):
            assert W.weight.shape[0] == mod.config.hidden_size and W.bias is None


TIED = None


@torch.no_grad()
def apply_edit(mod, d, lam):
    """W <- W - lam * d (d^T W) for every o_proj / down_proj; E <- E - lam (E d) d^T only if embeddings are not tied."""
    global TIED
    check_writers(mod)
    dv = torch.tensor(np.asarray(d, dtype=np.float32), device="cuda")
    dv = dv / dv.norm()
    for lyr in mod.model.layers:
        for W in (lyr.self_attn.o_proj.weight, lyr.mlp.down_proj.weight):
            W32 = W.data.float()
            W.data.copy_((W32 - lam * torch.outer(dv, dv @ W32)).to(W.dtype))
    E = mod.get_input_embeddings().weight
    TIED = bool(mod.get_output_embeddings() is not None and mod.get_output_embeddings().weight.data_ptr() == E.data_ptr())
    if not TIED:
        E32 = E.data.float()
        E.data.copy_((E32 - lam * torch.outer(E32 @ dv, dv)).to(E.dtype))
    return TIED


CHECK_TEXTS = [p["prompt"] for p in sorted(SETS["extraction"], key=lambda p: hashlib.sha256(p["prompt_id"].encode()).hexdigest())[:50]]


def proj_profile(mod, tok, d, Lsrc):
    """mean |act . d| at the last prompt token, for layers Lsrc..34 on the 50 fixed extraction prompts."""
    layers = list(range(Lsrc, mod.config.num_hidden_layers - 1))
    a = last_token_acts(mod, tok, CHECK_TEXTS, layers)
    dn = np.asarray(d, dtype=np.float32) / np.linalg.norm(d)
    return {L: float(np.abs(a[L] @ dn).mean()) for L in layers}


def build_edited(m, Lsrc, lam, directions, save_to):
    """Load model m, edit with fperp_{m}_L{Lsrc} at strength lam, run the projection check, save to save_to (local VM disk)."""
    mod, tok = hf_load(model_source(m))
    d = directions[f"fperp_{m}_L{Lsrc}"]
    before = proj_profile(mod, tok, d, Lsrc)
    tied = apply_edit(mod, d, lam)
    after = proj_profile(mod, tok, d, Lsrc)
    fall = {L: 1 - after[L] / before[L] for L in before}
    check = {"model": m, "Lsrc": Lsrc, "lam": lam, "tied_embeddings": tied, "embed_edited": not tied,
             "min_fall": float(min(fall.values())), "required": 0.9 * lam, "passed": bool(min(fall.values()) >= 0.9 * lam),
             "fall_by_layer": fall, "before": before, "after": after}
    print(f"  projection check {m} L{Lsrc} lam{lam}: min fall {check['min_fall']:.3f} (need {0.9 * lam:.2f}) -> {'pass' if check['passed'] else 'FAIL (logged)'}")
    import shutil
    if Path(save_to).exists():
        shutil.rmtree(save_to)
    mod.save_pretrained(save_to)
    tok.save_pretrained(save_to)
    del mod
    gc.collect()
    torch.cuda.empty_cache()
    return check


def load_directions(name="s2_directions"):
    d = OUT / name
    assert folder_ok(d), f"{name} not complete"
    z = np.load(d / "directions.npz")
    return {k: z[k] for k in z.files}, json.loads((d / "diagnostics.json").read_text())


def half(seed_id):
    return int(hashlib.sha256(seed_id.encode()).hexdigest(), 16) % 2 == 0


def judge_all(keys, by_key_prompt, by_key_answer, path, gid, util):
    """Judge (prompt, answer) for every key with one guard; resumes from the JSONL checkpoint at path."""
    v = {r["key"]: r for r in load_jsonl(path)}
    todo = [k for k in keys if k not in v]
    print(f"{Path(path).name}: {len(v)} judged, {len(todo)} to go")
    if todo:
        g = load_llm(gid, DTYPE, gpu_memory_utilization=util, max_model_len=8192)
        for i in range(0, len(todo), 800):
            ch = todo[i:i + 800]
            js = judge(g, [by_key_prompt[k] for k in ch], [by_key_answer[k] for k in ch])
            rs = [{"key": k, **j} for k, j in zip(ch, js)]
            append_jsonl(path, rs)
            v.update({r["key"]: r for r in rs})
        free_llm(g)
    return v


def stop_requested():
    return (IN / "STOP").exists() or (OUT / "STOP").exists()


In [ ]:
# XB2 step 1: one forward pass per model over all 3,864 extraction prompts; last prompt token for EVERY decoder layer.
# hidden_states[L+1] = output of decoder layer L (L = 0 .. num_hidden_layers-1). fp16, one file per model x layer.
from transformers import AutoConfig
ACTS = Path(ACTDIR)
ACTS.mkdir(parents=True, exist_ok=True)
P = SETS["extraction"]
NL = AutoConfig.from_pretrained(TINY_AYA).num_hidden_layers
ALL_LAYERS = list(range(NL))
print("num_hidden_layers:", NL)
idx = pd.DataFrame([{k: p[k] for k in ("prompt_id", "seed_id", "kind", "severity", "culture", "framing", "lang")} for p in P])
idx.to_json(ACTS / "index.jsonl", orient="records", lines=True, force_ascii=False)
NORM_INFO = {}
for m in MODELS:
    if all((ACTS / f"acts_{m}_L{L}.npy").exists() for L in ALL_LAYERS):
        print("activations present:", m)
        continue
    mod, tok = hf_load(model_source(m))
    assert mod.config.num_hidden_layers == NL
    # Is the last hidden state post final-norm? Compare hidden_states[NL] with the raw output of the last decoder layer on one prompt.
    cap = {}
    hk = mod.model.layers[-1].register_forward_hook(lambda mo, i, o: cap.__setitem__("o", (o[0] if isinstance(o, tuple) else o).detach()))
    with torch.no_grad():
        ids = torch.tensor([chat_ids(tok, P[0]["prompt"])]).cuda()
        o = mod(input_ids=ids, output_hidden_states=True, use_cache=False)
    hk.remove()
    NORM_INFO[m] = {"last_hidden_equals_raw_last_layer": bool(torch.allclose(o.hidden_states[NL].float(), cap["o"].float(), atol=1e-3)),
                    "n_hidden_states": len(o.hidden_states)}
    print(m, NORM_INFO[m])
    t0 = time.time()
    A = last_token_acts(mod, tok, [p["prompt"] for p in P], ALL_LAYERS, bs=BATCH)
    for L in ALL_LAYERS:
        np.save(ACTS / f"acts_{m}_L{L}.npy", A[L].astype(np.float16))
    del mod, A
    gc.collect()
    torch.cuda.empty_cache()
    print(f"activations done: {m} ({time.time() - t0:.0f}s)")
(ACTS / "norm_info.json").write_text(json.dumps(NORM_INFO, indent=1))
tot = sum(f.stat().st_size for f in ACTS.glob("acts_*.npy"))
print(f"activation files: {len(list(ACTS.glob('acts_*.npy')))}, total {tot / 1e9:.2f} GB")


In [ ]:
# XB2 step 2: XSTEER_SHARED section 2 definitions, identical for every model x layer.
from sklearn.metrics import roc_auc_score
idx = pd.read_json(ACTS / "index.jsonl", lines=True, dtype=False)
assert list(idx.prompt_id) == [p["prompt_id"] for p in P]
KIND, LANG, FRAME = (np.array(idx[c].values) for c in ("kind", "lang", "framing"))
TWIN = KIND == "twin"
FOLD = np.array([int(hashlib.sha256(s.encode()).hexdigest(), 16) % 2 for s in idx.seed_id])
# exact harmful/twin pairs: same seed, culture, framing, language; harmful severity clear
key = ["seed_id", "culture", "framing", "lang"]
Hs = idx[(idx.kind == "harmful") & (idx.severity == "clear")].reset_index().set_index(key)["index"]
Ts = idx[idx.kind == "twin"].reset_index().set_index(key)["index"]
assert Hs.index.is_unique and Ts.index.is_unique
common = Hs.index.intersection(Ts.index)
PH, PT = Hs.loc[common].values, Ts.loc[common].values
PLANG = LANG[PT]
# exact en/ja twin pairs: same seed, culture, framing
gkey = ["seed_id", "culture", "framing"]
TE = idx[(idx.kind == "twin") & (idx.lang == "en")].reset_index().set_index(gkey)["index"]
TJ = idx[(idx.kind == "twin") & (idx.lang == "ja")].reset_index().set_index(gkey)["index"]
assert TE.index.is_unique and TJ.index.is_unique
gcommon = TE.index.intersection(TJ.index)
GE, GJ = TE.loc[gcommon].values, TJ.loc[gcommon].values
print(f"harm pairs {len(PH)} (en {int((PLANG == 'en').sum())}, ja {int((PLANG == 'ja').sum())}); en/ja twin pairs {len(GE)}")

# refusal labels (g06_refused == 1 refused, == 0 answered, otherwise unknown) -- input manifests checked first
LABSRC = {"base": IN / "labels" / "steering_s1", "en_s0": IN / "labels" / "steering_s1", "bi_s0": IN / "labels" / "e1_s1",
          "ja_s0": OUT / "s1_labels_ja_s0"}
REF = {}
for m in MODELS:
    d = LABSRC[m]
    assert folder_ok(d), f"labels for {m} not complete: {d}"
    lab = pd.read_json(d / "answers.jsonl", lines=True, dtype=False)
    lab = lab[(lab.model == m) & (lab.kind == "twin")].set_index("prompt_id").g06_refused
    REF[m] = np.array(pd.to_numeric(idx.prompt_id.map(lab), errors="coerce"), dtype=float)
    REF[m][~TWIN] = np.nan
COUNTS = {m: {lang: {"refused": int(np.nansum((REF[m] == 1) & (LANG == lang))), "answered": int(np.nansum((REF[m] == 0) & (LANG == lang))),
                     "unknown": int(np.sum(TWIN & (LANG == lang) & np.isnan(REF[m])))} for lang in ("en", "ja")} for m in MODELS}
LOWN = {m: {lang: COUNTS[m][lang]["refused"] < 30 for lang in ("en", "ja")} for m in MODELS}
print(json.dumps(COUNTS))
print("low_n:", LOWN)


def unit(v):
    if v is None:
        return None
    n = float(np.linalg.norm(v))
    return v / n if n > 0 else None


def orth(x, basis):
    """x minus its projection on span(basis) (Gram-Schmidt), renormalised."""
    if x is None or any(b is None for b in basis):
        return None
    Q = []
    for b in basis:
        v = b.astype(np.float64).copy()
        for q in Q:
            v -= (v @ q) * q
        n = np.linalg.norm(v)
        if n > 1e-8:
            Q.append(v / n)
    y = x.astype(np.float64).copy()
    for q in Q:
        y -= (y @ q) * q
    return unit(y).astype(np.float32)


def cos(a, b):
    return None if a is None or b is None else float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


def fit(X, r, use):
    """All directions of section 2 fitted on the rows where `use` is True (a fold, or every seed)."""
    D, info = {}, {}
    s = use[PH]
    D["h"] = unit((X[PH[s]] - X[PT[s]]).mean(0)) if s.any() else None
    for lang in ("en", "ja"):
        sl = s & (PLANG == lang)
        D[f"h_{lang}"] = unit((X[PH[sl]] - X[PT[sl]]).mean(0)) if sl.any() else None
    gs = use[GE]
    D["g"] = unit((X[GE[gs]] - X[GJ[gs]]).mean(0)) if gs.any() else None
    tw = TWIN & use
    for lang in ("en", "ja"):
        num, wsum, strata = 0.0, 0, {}
        for st in ("control", "fc", "rp"):
            mm = tw & (LANG == lang) & (FRAME == st)
            rf, an = mm & (r == 1), mm & (r == 0)
            nr, na = int(rf.sum()), int(an.sum())
            used = nr >= 5 and na >= 5
            strata[st] = {"refused": nr, "answered": na, "used": used}
            if used:
                num = num + nr * (X[rf].mean(0) - X[an].mean(0))
                wsum += nr
        D[f"f_{lang}"] = unit(num / wsum) if wsum else None
        info[f"strata_{lang}"] = strata
    D["f_bal"] = unit(D["f_en"] + D["f_ja"]) if D["f_en"] is not None and D["f_ja"] is not None else None
    rf, an = tw & (r == 1), tw & (r == 0)
    D["f_pooled_old"] = unit(X[rf].mean(0) - X[an].mean(0)) if rf.any() and an.any() else None
    for lang in ("en", "ja"):
        D[f"f_{lang}_perp_h"] = orth(D[f"f_{lang}"], [D["h"], D[f"h_{lang}"]])
        D[f"f_{lang}_perp_hg"] = orth(D[f"f_{lang}"], [D["h"], D[f"h_{lang}"], D["g"]])
    for x in ("f_bal", "f_pooled_old"):   # two-language directions: h_lang = both h_en and h_ja
        D[f"{x}_perp_h"] = orth(D[x], [D["h"], D["h_en"], D["h_ja"]])
        D[f"{x}_perp_hg"] = orth(D[x], [D["h"], D["h_en"], D["h_ja"], D["g"]])
    D["f_pooled_old_perp_hsteer"] = orth(D["f_pooled_old"], [D["h"]])   # steering's f-perp (h only), for the saved-layer cross-check
    return D, info


POPS = {"en": TWIN & (LANG == "en"), "ja": TWIN & (LANG == "ja"), "all": TWIN}
AUC_DIRS = ["f_en", "f_en_perp_h", "f_en_perp_hg", "f_ja", "f_ja_perp_h", "f_ja_perp_hg", "f_bal", "f_bal_perp_h", "f_bal_perp_hg",
            "f_pooled_old", "f_pooled_old_perp_h", "f_pooled_old_perp_hg", "f_pooled_old_perp_hsteer", "g", "h", "h_en", "h_ja"]


def auc(X, d, r, mask):
    if d is None:
        return None
    m = mask & ~np.isnan(r)
    y = (r[m] == 1).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return None
    return float(roc_auc_score(y, X[m] @ d))


def nanmean2(a, b):
    v = [x for x in (a, b) if x is not None]
    return float(np.mean(v)) if len(v) == 2 else None


In [ ]:
# XB2 step 3: every direction and diagnostic for every model x layer (held-out = 2 seed folds, averaged; in-sample too).
OUTD = OUT / OUTNAME
OUTD.mkdir(parents=True, exist_ok=True)
ALLD, DIAG = {}, {}
t0 = time.time()
for m in MODELS:
    r = REF[m]
    for L in ALL_LAYERS:
        X = np.load(ACTS / f"acts_{m}_L{L}.npy").astype(np.float32)
        Dall, info = fit(X, r, np.ones(len(X), bool))
        folds = [fit(X, r, FOLD == k)[0] for k in (0, 1)]
        dd = {"model": m, "layer": L, **{f"n_{k}_{lang}": COUNTS[m][lang][k] for lang in ("en", "ja") for k in ("refused", "answered")},
              "low_n_en": LOWN[m]["en"], "low_n_ja": LOWN[m]["ja"], "strata_en": info["strata_en"], "strata_ja": info["strata_ja"],
              "act_norm_mean": float(np.linalg.norm(X, axis=1).mean())}
        for x in ("f_en", "f_ja", "f_bal", "f_pooled_old", "f_en_perp_hg", "f_ja_perp_hg", "f_bal_perp_hg"):
            dd[f"cos_{x}_h"] = cos(Dall[x], Dall["h"])
            dd[f"cos_{x}_g"] = cos(Dall[x], Dall["g"])
            for lang in ("en", "ja"):
                dd[f"cos_{x}_h_{lang}"] = cos(Dall[x], Dall[f"h_{lang}"])
        dd["cos_f_en_f_ja"] = cos(Dall["f_en"], Dall["f_ja"])
        dd["cos_f_en_perp_hg_f_ja_perp_hg"] = cos(Dall["f_en_perp_hg"], Dall["f_ja_perp_hg"])
        dd["cos_h_en_h_ja"] = cos(Dall["h_en"], Dall["h_ja"])
        dd["cos_h_g"] = cos(Dall["h"], Dall["g"])
        for dn in AUC_DIRS:
            for pop, pm in POPS.items():
                dd[f"auc_in_{dn}_{pop}"] = auc(X, Dall[dn], r, pm)
                dd[f"auc_ho_{dn}_{pop}"] = nanmean2(auc(X, folds[0][dn], r, pm & (FOLD == 1)), auc(X, folds[1][dn], r, pm & (FOLD == 0)))
        harm = (KIND == "harmful").astype(int)
        dd["auc_in_h_harmful_vs_twin"] = float(roc_auc_score(harm, X @ Dall["h"]))
        dd["auc_in_h_clear_pairs"] = float(roc_auc_score(np.r_[np.ones(len(PH)), np.zeros(len(PT))], np.r_[X[PH] @ Dall["h"], X[PT] @ Dall["h"]]))
        # steering-equivalent numbers (pooled f vs h; f-perp(h only) AUC on refused vs answered twins, in-sample) for the cross-check
        dd["steer_cos_f_h"] = cos(Dall["f_pooled_old"], Dall["h"])
        dd["steer_auc_fperp"] = dd["auc_in_f_pooled_old_perp_hsteer_all"]
        DIAG[f"{m}|L{L}"] = dd
        for k, v in Dall.items():
            if v is not None:
                ALLD[f"{k}__{m}__L{L}"] = np.asarray(v, dtype=np.float32)
        del X
    Lm = ALL_LAYERS[len(ALL_LAYERS) // 2]
    print(f"{m}: done ({time.time() - t0:.0f}s)  e.g. L{Lm}: AUC_ho f_en_perp_hg(en)={DIAG[f'{m}|L{Lm}']['auc_ho_f_en_perp_hg_en']}  "
          f"cos(f_en,h_en)={DIAG[f'{m}|L{Lm}']['cos_f_en_h_en']}")


def g_(k):
    return ALLD.get(k)


CROSS = {}
for L in ALL_LAYERS:
    c = {"layer": L}
    for a, b in [(("f_en", "en_s0"), ("f_ja", "ja_s0")), (("f_en", "en_s0"), ("f_en", "ja_s0")), (("f_ja", "en_s0"), ("f_ja", "ja_s0")),
                 (("f_en", "en_s0"), ("f_en", "bi_s0")), (("f_ja", "ja_s0"), ("f_ja", "bi_s0")), (("f_en", "en_s0"), ("f_en", "base")),
                 (("f_en_perp_hg", "en_s0"), ("f_ja_perp_hg", "ja_s0")), (("h", "en_s0"), ("h", "ja_s0")), (("h", "en_s0"), ("h", "base"))]:
        c[f"cos_{a[0]}[{a[1]}]_{b[0]}[{b[1]}]"] = cos(g_(f"{a[0]}__{a[1]}__L{L}"), g_(f"{b[0]}__{b[1]}__L{L}"))
    for m in MODELS:
        c[f"cos_h_en_h_ja[{m}]"] = DIAG[f"{m}|L{L}"]["cos_h_en_h_ja"]
    CROSS[f"L{L}"] = c


In [ ]:
# XB2 step 4: write directions_all.npz, diagnostics.json/csv, cross-model cosines; manifest; DONE.
np.savez(OUTD / "directions_all.npz", **ALLD)
(OUTD / "diagnostics.json").write_text(json.dumps({"diag": DIAG, "cross_model": CROSS, "counts": COUNTS, "low_n": LOWN, "layers": ALL_LAYERS,
                                                   "num_hidden_layers": NL, "n_harm_pairs": int(len(PH)), "n_lang_pairs": int(len(GE)),
                                                   "norm_info": json.loads((ACTS / "norm_info.json").read_text()) if (ACTS / "norm_info.json").exists() else {}},
                                                  indent=1))
flat = pd.DataFrame([{k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in d.items()} for d in DIAG.values()])
flat.to_csv(OUTD / "diagnostics.csv", index=False)
pd.DataFrame(CROSS.values()).to_csv(OUTD / "cross_model.csv", index=False)
act_files = sorted(f.name for f in ACTS.glob("acts_*.npy"))
publish(OUTD, ["directions_all.npz", "diagnostics.json", "diagnostics.csv", "cross_model.csv"],
        {"stage": OUTNAME, "models": MODELS, "layers": ALL_LAYERS, "n_prompts": len(P), "counts": COUNTS, "low_n": LOWN,
         "activations": {"dir": str(ACTS), "n_files": len(act_files), "total_bytes": int(sum((ACTS / f).stat().st_size for f in act_files)),
                         "sha256": {f: sha256_file(ACTS / f) for f in act_files} if HASH_ACTS else None},
         "label_sources": {m: str(LABSRC[m]) for m in MODELS},
         "label_manifest_sha256": {m: sha256_file(LABSRC[m] / "manifest.json") for m in MODELS},
         "adapter_sha256": {m: {f: sha256_file(IN / "adapters" / m / f) for f in ("adapter_config.json", "adapter_model.safetensors")} for m in MODELS if m != "base"},
         "base": {"id": TINY_AYA, "sha": model_sha(TINY_AYA)}})
print("published", OUTD)
